In [ ]:
# Скрипт для собирания полных данных из статей блогов (полный текст, тэги)

# 22.11 9:23 "Чистый" скрипт, в новом файле под старым названием fetching_full_article

In [ ]:
import time
import math
import random
import requests
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import datetime, date, timezone
import re
import psycopg2
from psycopg2.extras import execute_values

from google.colab import userdata
neon_db_pass = userdata.get('neon_db_pass')

In [ ]:
#|%%%%%%%%%%%%%%%%%%%%%%%%%%| ПАРАМЕТРЫ БД |%%%%%%%%%%%%%%%%%%%%%%%%%%|
DB_PARAMS = {
    "host": "ep-flat-feather-b1mwouhw-pooler.c-5.eu-central-1.aws.neon.tech",
    "database": "neondb",
    "user": "neondb_owner",
    "password": neon_db_pass,
    "port": 5432,
    "sslmode": "require"
}

TABLE_NAME_ARTICLES = 'articles_briefs'
TABLE_NAME_BLOGS = 'blogs'
TABLE_NAME_FULL_ARTICLES = 'articles_content'
TABLE_NAME_TAGS = 'tags'

#|%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%|


# |%%%%%%%%%%%%%%%%%%%%%%%%%% ПАРАМЕТРЫ ПАРСЕРА %%%%%%%%%%%%%%%%%%%%%%%%%%|
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Referer': 'https://www.sports.ru/'
}

ARTICLES_TO_FETCH = 50
# |%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%|


In [ ]:
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-= #
#   ------------  D B   F U N C T I O N S  --------------   #
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-= #

# "базового" урла нам здесь не понадобится - будем забирать урл из БД
# могли бы использовать только id, но так чуть удобнее...

def GetLastNUrls(db_params, table_name, n_articles) -> list:
    '''
    Забираем ещё не обработанные брифы - то есть те, в которых флаг has_full_text
    стоит в фолс
    '''
    print(f'Подключаемся к таблице {table_name}, чтобы забрать {n_articles} записей без полного текста...')
    try:
        conn = psycopg2.connect(**db_params)
        cursor = conn.cursor()
        sql_query = f'''
        SELECT article_id, url FROM {table_name}
        WHERE has_full_text = FALSE
        ORDER BY published_at ASC
        LIMIT {n_articles}
        '''
        cursor.execute(sql_query)
        raw_data = cursor.fetchall()
        print(f'☑ Успешно получено записей: {len(raw_data)}')
    except Exception as e:
        print(f'Не удалось подключиться или выполнить запрос в БД {table_name}. Ошибка {e}')
        return None
    finally: # при любом исходе - дверь за собой закрываем
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()

    return raw_data


# Вытащил проверку тэгов на новизну и добавление их в отд. функцию, больно здоровая выходит
def AddingNewTags(tags_json):
    """
    Определяем, есть ли среди присланных тэгов новые (напом., в "тэге" 3 составляющих -
    имя, урл, урл картинки, уник определяется урлом), и вносим в таблицу, если естьчо.
    На вход: список из тьюплов словарей - тэгов (в виде джсона), т.е. ({тут тэги статьи}, )

    На выходе: айдишники добавленных тэгов or ничего or False если были траблы
    """

    # START --- под-функция для сплитования урла тэга -----
    def splitting_tag_url(url) -> tuple:
        """
        разбиваем урл тэга (стандартно на спортсе выглядит как
        'football/club/diyarbak-r-buyuksehir-belediyespor/')
        на 3 части: имя, тип сущности, вид спорта
        Возвращаем: тьюпл сих вещей или все Ноны
        """
        import re

        url_parts = re.findall(r'[^/]+', url)
        eng_name, entity_type, entity_sport = None, None, None
        if not url_parts:
            print(f'ВАЖНО: не сумели выделить части из урла тэга {url}')
            return (None, None, None)
        if len(url_parts) < 3:
            return (None, None, None)
        # наивно предполагаем, что в таком случае изменения будут в начале, а не с конца, и не меняем структуру ответа
        if len(url_parts) > 3:
            print(f'''ВАЖНО: Структура урла для тэгов, прописанная в вёрстке статей, изменилась,
            частей между слэшами больше трёх и они м.б. выделены неправильно. УРЛ: {url}')
            ''')

        eng_name = url_parts[-1]
        entity_type = url_parts[-2]
        entity_sport = url_parts[-3]
        return (eng_name, entity_type, entity_sport)
    # END --- под-функция для сплитования урла тэга -----

    try:
        conn = psycopg2.connect(**DB_PARAMS)
        cursor = conn.cursor()
        print('Успешно подключились к БД для работы над тэгами.')
        # определение новых тэгов. дюже сложный sql, не стал возиться с pandas и попросил Иишку, как сделать в sql:)
            # Запрос, который найдет только те теги, которых еще НЕТ в таблице tags
        find_new_tags_query = f"""
            WITH incoming_json AS (
                -- 1. Принимаем пачку данных из Python в виртуальную таблицу data_source
                SELECT t_json::JSONB AS raw_json
                FROM (VALUES (%s::TEXT)) AS data_source (t_json)
            ),
            expanded_tags AS (
                -- 2. «Распаковываем» массивы тегов во множество строк (как explode в Pandas)
                SELECT
                    (jsonb_array_elements(raw_json)->>'tag_name') AS name,
                    (jsonb_array_elements(raw_json)->>'tag_url') AS url,
                    (jsonb_array_elements(raw_json)->>'tag_img') AS img
                FROM incoming_json
            )
            -- 3. Делаем LEFT JOIN со справочником tags по URL.
            -- Если в БД тега нет, t.tag_id вернет NULL — это и есть наши "новички"!
            SELECT DISTINCT eth.name, eth.url, eth.img
            FROM expanded_tags AS eth
            LEFT JOIN {TABLE_NAME_TAGS} AS t ON t.tag_url = eth.url
            WHERE t.tag_id IS NULL; -- Фильтруем: только те, которых нет в базе
        """

        execute_values(cursor, find_new_tags_query, tags_json)

        # Забираем только новые теги, которых нет в справочнике
        new_tags_found = cursor.fetchall()
        if new_tags_found:
            print(f"...найдены новые тэги в количестве: {len(new_tags_found)}")
            print("...пример тега:", new_tags_found[0][0])

            # добавляем к тьюплу с инфой по новому тэгу инфу, взятую из его урла!
            expanded_new_tags = [item + splitting_tag_url(item[1]) for item in new_tags_found]

            # фух, всё, теперь можно заносить в БД
            add_new_tags_query = f'''
                INSERT INTO {TABLE_NAME_TAGS}
                (
                    tag_name, tag_url, tag_image_url, eng_name, entity, sport
                )
                VALUES %s
                ON CONFLICT (tag_url) -- по идее, быть не должно, мы ж только что уникальные отфильтровали...
                DO NOTHING
                RETURNING tag_id; -- ID тэгов, которые вставились
            '''
            execute_values(cursor, add_new_tags_query, expanded_new_tags)
            inserted_tids = [row for row in cursor]

            conn.commit()
            print('...новые тэги внесены в БД, айдишники им назначены!')
        else:
            return

    except Exception as e:
        print(f"Проблемы при поиске\добавлении новых тэгов в спарсенной информации. Ошибка: {e}")
        return False
    finally:
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()

    return inserted_tids



def SavingFullArticles_DB(fetched_articles, db_params, table_name):
    '''
    а) записываем полные данные статей в БД
    б) в табл. брифов проставляем флаг обработанности вставленных статей
       на выходе: список айдишников статей, которые мы залили в таблицу. Если новых не добавлено
       - будет пустой список
    '''
    # --------- поиск НОВЫХ ТЭГов ------------
    # если уже тут проблемы - остальное вносить не надо, будем выходить

    # тэги у нас в 4-ом айтеме присылаемой инфы. разворачиваем так, чтобы
    # каждый тэг был в собственном тьюпле, из какой статьи он - не важно
    tags_to_check = [(item[4], ) for item in fetched_articles]
    # если всё норм, в след. переменной будет список вставленных id_tags или пустой лист (нет новых тэгов), False там м.б. только в случае траблов
    is_tag_ok = AddingNewTags(tags_to_check)
    if is_tag_ok == False:
        print('КРИТИЧНО: Возникли проблемы в процессе проверки тэгов на новизну, все данные по статьям добавлены не будут')
        return False


    try:
        conn = psycopg2.connect(**db_params)
        cursor = conn.cursor()
        print(f"Успешно подключились к БД для записи статей в таблицу {table_name}")

        # --------- МАТЧИМ каждый ТЭГ с айди статьи и заносим в кросс-таблицу ------------
        # тоже суровый sql, повозился сам, да спросил у иишки...
        tag_article_linking_query = '''
            WITH data_source AS (
                SELECT article_id, t_json::JSONB AS raw_json
                FROM (VALUES %s) AS ds (article_id, full_text, chars_count, words_count, t_json, images_count, images_urls)
            ),
            expanded_tags AS (
                SELECT
                    ds.article_id,
                    j->>'tag_name' AS name,
                    j->>'tag_url'  AS url,
                    j->>'tag_img'  AS img
                FROM data_source AS ds
                CROSS JOIN LATERAL jsonb_array_elements(ds.raw_json) AS j
            )
            SELECT DISTINCT
                eth.article_id,
                t.tag_id
            FROM expanded_tags AS eth
            JOIN tags AS t ON t.tag_url = eth.url;
            '''
        execute_values(cursor, tag_article_linking_query, fetched_articles)
        tags_results = [row for row in cursor]
        print(f'...вставлено {len(tags_results)} соответствий |id_статьи-id_тэга| в таблицу связей.')

        # ------------ ВСТАВКА непосредственно, самое простое :) ---------------
        query = f"""
            INSERT INTO {table_name} (
                article_id, full_text, chars_count, words_count, tags, images_count, images_urls
            )
            VALUES %s
            ON CONFLICT (article_id)
            DO NOTHING
            RETURNING article_id; -- База вернет ID только тех строк, которые РЕАЛЬНО вставились;
        """
        execute_values(cursor, query, fetched_articles)
        # Вычерпываем ID из курсора прямо во время/после выполнения execute_values.
        # Так как база возвращает кортежи вида (3550494,), берем [0] элемент.
        inserted_ids = [row for row in cursor]
        real_inserted = len(inserted_ids)

        print(f"Всего пришло записей: {len(fetched_articles)}. Вставлено новых записей: {real_inserted}")

# -- вставили! ТЕПЕРЬ НУЖНО ОТМЕТИТЬ В ТАБЛИЦЕ БРИФОВ ЭТИ ID КАК СОХРАНЁННЫЕ! ----
# апд. сперва изменял флаг has_full_text только у inserted_ids. МИНУС в том, что если уже оказалась статья в сохранённых, то флаг её не будет меняться,
# ибо айди не будет в inserted_ids... как такое возможно: может происходить при тестовых, ручных добавлениях в бд, при добавлении данных из джсона _вне этого_ парсера... поэтому просто все айдишники пришедшие отмечаем, так надёжнее
        all_incoming_ids = [(post[0],) for post in fetched_articles]

        if all_incoming_ids:
            # print(type(inserted_ids[0]))
            print('Изменяем флаг обработки в таблице брифов статей...')
            query = f'''
            UPDATE articles_briefs AS b
            SET has_full_text = TRUE
            FROM (VALUES %s) AS data_source (t_id)
            WHERE b.article_id = data_source.t_id::BIGINT;
            '''
            execute_values(cursor, query, all_incoming_ids)
        else:
            print('Новых статей не было, в таблице брифов апдейтить нечего...')

        # проводим и вставку, и обновление флагов
        conn.commit()
        # print('★ Все операции с БД выполнены успешно!')

    # любые проблемы с записью в БД приведут сюда. поэтому здесь нужно выдавать какой-то возвращаемый флаг о проблемах - просто пустой список не пойдёт, ведь он значит также, что просто не было новых статей...
    # ... а, придумал! можно спрашивать, какой тип того, что нам вернулось: если список, то всё Ок, если boolean - надо включить запасной план, ведь с бд не получилось...
    except Exception as e:
        print(f"Произошла ошибка: {e}")
        if conn:
            conn.rollback()
        return False

    finally:
        if 'cursor' in locals(): cursor.close()
        if 'conn' in locals(): conn.close()

    return [id[0] for id in inserted_ids]

In [ ]:
def SavingFullArticles_JSON(fetched_articles, filename='full_articles_data'):

    # обеспечиваем уникальное имя файла - даты-времени хватит )
    unique_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = filename + '_' + unique_suffix + '.json'

    try:
        with open(fname, 'w') as f:
            json.dump(fetched_articles, f, ensure_ascii=False, indent=4)
        print(f'--- Данные успешно сохранены в файл {fname} ---')
    except Exception as e:
        print('КРИТИЧНО: Локально тоже не получилось сохранить спарсенные данные :(. Ошибка:', e)
        FailedToSave((fetched_articles[0][0], fetched_articles[-1][0]), 'Встряли после парсинга полных текстов статей из блогов')

In [ ]:
def FailedToSave(first_last, msg='we have failed to save data...'):
    '''
    Эта функция вызывается _только_ если всё плохо при записи - не вышло сохранить в БД и не вышло
    сохранить локально в json.
    На входе - тьюпл из первого и посл. id в несохранённых данных.
    msg - сообщение, которое будем отсылать мне - можно прописать, откуда именно вызывали Функцию Адского Дня
    '''
    print('%'*60)
    print('Вас беспокоит ФУНКЦИЯ АДСКОГО ДНЯ! Если меня вызывали, значит, парсеры перестали работать или никак не получилось сохранить спарсенные данные...')
    print('Первый и последний id', first_last)
    print('Присланный комментарий:', msg)
    print('%'*60)
    return


In [ ]:
def clean_parsed_news(text):

    text = " ".join(text.split())
    return text

def parse_fullarticle_page(bs_tree, art_url):
    '''
    берём статью в виде объекта beautiful soup и выбираем из него вкусное.
    на выходе: tuple из (полн.текст - строка, тэги - json, кол-во картинок - инт, урлы картинок - json)
    '''

    # -------  РАЗБИРАЕМ ТЕКСТ ------- #

    if bs_tree.find('div', class_='document-header__title'):
        art_title = bs_tree.find('div', class_='document-header__title').text

    full_article = bs_tree.find('div', class_="post-body")

    if not full_article:
        print(f'КРИТИЧНО: тэгов полной статьи не обнаружно на странице {art_url}, выходим')
        return

    # разбираем весь текст по параграфам (.get_text() не юзаем, в новостях попадались карточки матчей)
    full_text_html = full_article.find_all('p', class_='sb-paragraph sb-paragraph--regular')

    full_text = ''
    for paragraph in full_text_html:
        full_text += paragraph.text + ' '

    # отделяем заголовок (я поставил просто перенос, иишка посоветовала добавить второй и чуток маркдауна:)
    full_text = '# '+ art_title + '\n\n' + clean_parsed_news(full_text)

    chars_count = len(full_text)
    words_count = len(full_text.split())

    # ------- РАЗБИРАЕМ КАРТИНКИ ------- #

    images_count = None
    images_json = None
    all_pics_html = full_article.find_all('figure', class_="sb-image")
    url_pics = []
    if all_pics_html:
        for pic in all_pics_html:
            if pic.find('img'):
                a_pic_url = pic.find('img').get('src')
                # нам нужны только картинки для статьи, всякая реклама - не нужна.
                # пока "статейные" Pics начинались только так: photobooth.cdn.sports.ru/ подобное и ищем
                url_matching = re.search(r"/([\w.-]+)/", a_pic_url)
                            # dumpster? но не реклама - "https://dumpster.cdn.sports.ru/3/17/3751884fa9230088d72e7c5b5a719.jpg?wу=webp&q=90&s=2x&w=730"
                            # dumpster, реклама 'https://dumpster.cdn.sports.ru/0/4a/c13d826d04dd1732c822773878fa1.png?q=90'
                # есть норм. картинки и в dumpster. на основе нескольких десятков примеров видим, что все не рекламные имеют доп. параметры, кроме q=90
                # будем искать только вхождение w=
                url_matching_w = re.search(r"([?&]w=)", a_pic_url)
                if url_matching:
                    if ('photobooth' in url_matching.group(1)) or url_matching_w:
                        url_pics.append(a_pic_url)
                    else:
                        print('WARNING: Урл кажется рекламной картинкой, не добавляем:', a_pic_url)
            else:
                print(f'ВАЖНО: не найден тэг img там, где предполагается картинка! Проверьте тэг figure в тексте статьи (class_="post-body") на странице {art_url}')
                continue
        # 1. Считаем количество для ML-фичи
        images_count = len(url_pics)
        # 2. Переводим список в JSON-строку для базы
        images_json = json.dumps(url_pics, ensure_ascii=False)

    # ------- РАЗБИРАЕМ ТЭГИ ------- #

    tags_list = []
    # все тэги на конкр. странице
    all_tags = bs_tree.find('div', class_ = 'tags-list')

    if not all_tags:
        print(f'ЭТОНОРМ: Новостных тэгов на странице с id={art_url} не нашлось...')
    else:
        for tag in all_tags.find_all('div', class_="tags-list-item"):
            one_tag = tag.find('a')
            # а вдруг тэг есть, но ссылкой не обзавёлся? предусмотрим!
            if one_tag:
                tag_name = one_tag.find('span')
                tag_img = one_tag.find('img')
                tag_dict = {
                    'tag_name': tag_name.text if tag_name else None,
                    'tag_url': one_tag.get('href'),
                    'tag_img': tag_img.get('src') if tag_img else None,
                }
                tags_list.append(tag_dict)
            else:
                print(f'У новостного тэга на странице {art_url} нет гиперссылки')
        # Это апдейт - сразу переводим в json, чтобы беспроблемно закинуть в БД
        tags_list = json.dumps(tags_list, ensure_ascii=False)

    return (full_text, chars_count, words_count, tags_list, images_count, images_json)

In [ ]:
# ----- вспомогательный генератор батчей -----

def generating_batches(lst: list, n_batches: int = 20):
    '''
    разбиваем входящий список на n_batches кусков (по дефолту, 20)
    выдаём назад в режиме генератора
    '''
    for i in range(0, len(lst), n_batches):
        yield lst[i : i+n_batches]


# ----- Функция для забирания урлов и отправления их в парсер -----

def urls_pages_batch_processing(urls_for_fetching:list, pref_batch=20)-> list:
    '''
    В простом цикле перебираем урлы новостей, коннектимся к каждому и спарсиваем полный текст + тэги
    start_url: для универсальности. сейчас парсер заточен на спортс, но можно прокидывать этот урл в парсер и менять тэги\классы под чемпионат.ком, например
    pref_batch: предпочтительный (потому что выборка м.б. меньше батча в реале) размер батча.

    на вход принимаем _список тьюплов_ id-url типа [(3485772, 'https://www.sports.ru/football/blogs/3485772.html') ...]
    на выходе: список тьюплов с распарсенными данными:
        article_id, полный текст статьи, сколько букофф?, сколько слов?, тэги - дикт, сколько картинок?, урлы картинок - список
    '''

    all_full_articles = []
    # счётчики. особо не нужны, но пусть остаются
    articles_parsed = 0
    missed_articles = 0
    pages_downloaded = 0
    missed_pages = 0
    total_batches_num = 1 + len(urls_for_fetching) // pref_batch

    print('='*27, 'Запускаем парсер...', '='*27)

    start_time_get = time.perf_counter()

    with requests.Session() as session:
        session.headers.update(headers)

    # в этот раз забабахали собств. генератор, так проще будет разбивать на батчи :)
    for b_count, batched_ids in enumerate(generating_batches(urls_for_fetching, pref_batch)):
        # весь enumerate нужен только чтобы не сидеть паузу после самого посл. батча (и не вводить доп проверки и переменные, чтобы это побороть, как в прошлом скрипте я делал)
        if b_count > 0:         # блок врубается со второго батча
            # между батчами отдыхаем дольше
            delay = random.uniform(15, 30)
            time.sleep(delay)

        for url in batched_ids:
            try:
                response = session.get(url[1], timeout=7)
                if response.status_code == 200:
                    # print(f"[НАЙДЕНО] Страница: {url[1]}")
                    pages_downloaded += 1
                else:
                    print(f"БЫВАЕТ: Не смогли зайти на страницу: {url[1]}, статус: {response.status_code}")
                    missed_pages += 1
                    # немного подождём в таким случае, прежде чем продолжать...
                    delay = random.uniform(11, 15)
                    time.sleep(delay)
                    continue

                tree = BeautifulSoup(response.content, 'html.parser')
                one_full_article = parse_fullarticle_page(tree, url[1])
                # если новость (1-й элемент возврата) не нашли на этой странице, выходим
                if not one_full_article:
                    print(f'ВАЖНО: Не нашли статьи на странице {url[1]}')
                    missed_articles += 1
                    continue
                # пробрасываем id статьи далее по пайплайну, будет оченно нужен!
                one_full_article = (url[0], *one_full_article)
                all_full_articles.append(one_full_article)
                articles_parsed += 1

            except requests.exceptions.RequestException as e:
                print(f"Ошибка при запросе {url[1]}: {e}")
                missed_pages += 1
                continue

            # рандомная пауза между урлами в батче
            delay = random.uniform(6, 10)
            time.sleep(delay)

        # небольшой лог после батча, чтобы было видно, что работаем...
        print(f'Обработано урлов: {len(batched_ids)}, батч № {b_count+1} из {total_batches_num}')


    end_time_get = time.perf_counter()
    duration_get = end_time_get - start_time_get
    print('-'*30)
    print(f'''Парсинг занял {duration_get:.0f} сек.
Страниц открыто: {pages_downloaded}, страниц не удалось открыть: {missed_pages},
    обработано статей: {articles_parsed}, не найдено статей на страницах: {missed_articles}''')
    print('='*75)

    return all_full_articles

In [ ]:
def main():
    # забираем последние N необработанных брифов
    articles_to_parse = GetLastNUrls(DB_PARAMS, TABLE_NAME_ARTICLES, ARTICLES_TO_FETCH)
    if len(articles_to_parse) < 1:
        print('Обрабатывать нечего! Отдыхаем.')
        return

    # ...отправляем их в ПАРСЕР
    arts = urls_pages_batch_processing(articles_to_parse)

    if len(arts) > 0:
        # ...потыренное надо схоронить!
        start_time_get = time.perf_counter()
        inserted_art_ids = SavingFullArticles_DB(arts, DB_PARAMS, TABLE_NAME_FULL_ARTICLES)

        # не в облака, так в личные закрома...
        # если на выходе имеем не список - значит только что сработало исключение и ничего не вышло
        if isinstance(inserted_art_ids, list):
            print('☑ Всё прошло успешно. Самый свежий вставленный id:', max(inserted_art_ids, default=-1))
            print(f'Работа с базой данных заняла {(time.perf_counter() - start_time_get):.1f} сек.')
        else:
            print('ВАЖНО: Сохранение в БД не получилось, пытаемся сохранить в JSON...')
            SavingFullArticles_JSON(arts)
    return arts


if __name__ == '__main__':
    rr = main()

Подключаемся к таблице articles_briefs, чтобы забрать 75 записей без полного текста...
☑ Успешно получено записей: 15
=========================== Запускаем парсер... ===========================
БЫВАЕТ: Не смогли зайти на страницу: https://www.sports.ru/football/blogs/3628233.html, статус: 404
Обработано урлов: 15, батч № 1 из 1
------------------------------
Парсинг занял 139 сек.
Страниц открыто: 14, страниц не удалось открыть: 1,
    обработано статей: 14, не найдено статей на страницах: 0
Успешно подключились к БД для работы над тэгами.
Успешно подключились к БД для записи статей в таблицу articles_content
...вставлено 109 соответствий |id_статьи-id_тэга| в таблицу связей.
Всего пришло записей: 14. Вставлено новых записей: 14
Изменяем флаг обработки в таблице брифов статей...
☑ Всё прошло успешно. Самый свежий вставленный id: 3652763
Работа с базой данных заняла 2.9 сек.


In [ ]:
'''
22.11 УЖЕ НЕСКОЛЬКО РАЗ НЕТ КОННЕКТА К ОДНОЙ НОВОСТИ, ВИДИМО УДОЛИЛИ

Надо сделать штуку, чтобы не лезть в такое постоянно.
После каждого прогона - добавляем проблемные урлы в отд. таблицу и записываем туда счётчик, и + 1, если там урлы такие есть

И если счётчик превысил цифру 5 - "вычёркиваем" его из таблицы брифов, в поле full text можно записать чтонить вроде
"недоступная статья по результатам N проверок" (главное чтобы не NULL, чтобы айди не подхватывался для проверки в след. заход).
апд. А, не, у меня там флаг... WHERE has_full_text = FALSE

То есть надо наверное добавить ещё поле типа deleted или inaccessible, и собирать статьи на фуллтекст_парс с условие WHERE has_full_text = FALSE AND deleted = FALSE
'''